# Marketing Campaign Performance Analysis

A portfolio project analyzing marketing channels, customer engagement, conversion behavior, advertising spend, and campaign efficiency.

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

# --- Visual style: one consistent theme, but a distinct color role per chart type,
# so ranked bars, category bars, trend lines, and comparisons don't all look the same ---
sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titlepad'] = 12
plt.rcParams['figure.facecolor'] = 'white'

CATEGORICAL = sns.color_palette("Set2")                  # distinct hues: unordered categories
SEQUENTIAL  = sns.color_palette("crest", as_cmap=True)    # gradient: ranked / sorted bars
ACCENT_A    = "#2E6F95"   # primary trend line
ACCENT_B    = "#E07A5F"   # secondary trend line
GOOD_BAD    = ["#588157", "#BC4749"]   # semantic pair: positive vs negative outcome

def add_bar_labels(ax, fmt="{:,.0f}"):
    for p in ax.patches:
        h = p.get_height()
        if pd.notna(h):
            ax.annotate(fmt.format(h), (p.get_x() + p.get_width() / 2, h),
                        ha='center', va='bottom', fontsize=9, color='#333333',
                        xytext=(0, 3), textcoords='offset points')

## 2. Load Dataset

In [2]:
df = pd.read_csv("marketing_campaign_data.csv")
df.head()

,Lead_ID,Channel,Age,Monthly_Income,Website_Visits,Ad_Clicks,Ad_Spend,Previous_Purchases,Device,Converted
0,300001,Social Media,27,11145.10,4,2,327.84,4,Mobile,0
1,300002,Email,54,13787.60,13,6,159.80,2,Desktop,0
2,300003,Social Media,53,11333.17,15,5,96.61,0,Mobile,0
3,300004,Referral,54,3492.53,6,2,323.26,3,Desktop,0
4,300005,Email,-5,11726.26,13,2,107.22,2,Mobile,0


## 3. Data Understanding

In [3]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nData types:")
print(df.dtypes)

Shape: (6600, 10)

Missing values:
Lead_ID                0
Channel               78
Age                    0
Monthly_Income        65
Website_Visits         0
Ad_Clicks              0
Ad_Spend               0
Previous_Purchases     0
Device                69
Converted              0
dtype: int64

Duplicate rows: 100

Data types:
Lead_ID                 int64
Channel                object
Age                     int64
Monthly_Income        float64
Website_Visits          int64
Ad_Clicks               int64
Ad_Spend              float64
Previous_Purchases      int64
Device                 object
Converted               int64
dtype: object


In [4]:
df.describe(include='all')

,Lead_ID,Channel,Age,Monthly_Income,Website_Visits,Ad_Clicks,Ad_Spend,Previous_Purchases,Device,Converted
count,6600.000000,6522,6600.000000,6535.000000,6600.000000,6600.000000,6600.000000,6600.000000,6531,6600.000000
unique,NaN,5,NaN,NaN,NaN,NaN,NaN,NaN,3,NaN
top,NaN,Social Media,NaN,NaN,NaN,NaN,NaN,NaN,Mobile,NaN
freq,NaN,1594,NaN,NaN,NaN,NaN,NaN,NaN,3634,NaN
mean,303202.015152,NaN,41.210909,8490.478395,7.965606,2.677424,199.402144,3.036212,NaN,0.193485
std,1902.852928,NaN,14.228636,3084.886161,4.306685,1.601552,127.600873,1.996145,NaN,0.395060
min,300001.000000,NaN,-5.000000,2000.000000,1.000000,0.000000,-186.710000,0.000000,NaN,0.000000
25%,301550.750000,NaN,29.000000,6379.420000,4.000000,1.000000,107.037500,1.000000,NaN,0.000000
50%,303200.500000,NaN,41.000000,8488.580000,8.000000,3.000000,174.225000,3.000000,NaN,0.000000
75%,304850.250000,NaN,53.000000,10588.130000,12.000000,4.000000,263.652500,5.000000,NaN,0.000000


## 4. Data Cleaning

In [5]:
df = df.drop_duplicates().reset_index(drop=True)

for col in ['Channel','Device']:
    df[col] = df[col].astype('string').str.strip().str.title()

df['Channel'] = df['Channel'].fillna('Unknown')
df['Device'] = df['Device'].fillna('Unknown')

df.loc[(df['Age'] < 18) | (df['Age'] > 100), 'Age'] = np.nan
df['Age'] = df['Age'].fillna(df['Age'].median())

for col in ['Monthly_Income','Ad_Spend']:
    df.loc[df[col] < 0, col] = np.nan
    df[col] = df[col].fillna(df[col].median())

for col in ['Website_Visits','Ad_Clicks','Previous_Purchases']:
    df[col] = df[col].clip(lower=0)

print(df.isnull().sum())

Lead_ID               0
Channel               0
Age                   0
Monthly_Income        0
Website_Visits        0
Ad_Clicks             0
Ad_Spend              0
Previous_Purchases    0
Device                0
Converted             0
dtype: int64


## 5. Feature Engineering

In [6]:
df['Conversion_Rate'] = np.where(
    df['Website_Visits'] > 0,
    df['Converted'] / df['Website_Visits'] * 100,
    0
).round(2)

df['Engagement_Level'] = pd.cut(
    df['Ad_Clicks'],
    bins=[-1,1,3,6,np.inf],
    labels=['Low','Medium','High','Very High']
)

df['Income_Band'] = pd.qcut(
    df['Monthly_Income'], 4,
    labels=['Low','Mid-Low','Mid-High','High']
)

df.head()

,Lead_ID,Channel,Age,Monthly_Income,Website_Visits,Ad_Clicks,Ad_Spend,Previous_Purchases,Device,Converted,Conversion_Rate,Engagement_Level,Income_Band
0,300001,Social Media,27.0,11145.10,4,2,327.84,4,Mobile,0,0.0,Medium,High
1,300002,Email,54.0,13787.60,13,6,159.80,2,Desktop,0,0.0,High,High
2,300003,Social Media,53.0,11333.17,15,5,96.61,0,Mobile,0,0.0,High,High
3,300004,Referral,54.0,3492.53,6,2,323.26,3,Desktop,0,0.0,Medium,Low
4,300005,Email,41.0,11726.26,13,2,107.22,2,Mobile,0,0.0,Medium,High


## 6. Exploratory Data Analysis

In [ ]:
channel_conversion = df.groupby('Channel')['Converted'].mean().mul(100).sort_values(ascending=False)

colors = SEQUENTIAL(np.linspace(0.25, 0.9, len(channel_conversion)))
plt.figure(figsize=(8, 4.5))
ax = plt.gca()
ax.bar(channel_conversion.index, channel_conversion.values, color=colors, edgecolor='white', linewidth=0.6)
ax.set_title('Conversion Rate by Marketing Channel')
ax.set_ylabel('Conversion Rate (%)')
plt.xticks(rotation=20)
add_bar_labels(ax, "{:.1f}%")
plt.tight_layout()
plt.show()

In [ ]:
channel_spend = df.groupby('Channel')['Ad_Spend'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4.5))
ax = plt.gca()
ax.bar(channel_spend.index, channel_spend.values, color=CATEGORICAL[:len(channel_spend)],
       edgecolor='white', linewidth=0.6)
ax.set_title('Advertising Spend by Channel')
ax.set_ylabel('Ad Spend')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
plt.xticks(rotation=20)
add_bar_labels(ax, "{:,.0f}")
plt.tight_layout()
plt.show()

In [ ]:
engagement_conversion = df.groupby('Engagement_Level', observed=True)['Converted'].mean().mul(100)

plt.figure(figsize=(7, 4.5))
ax = plt.gca()
ax.plot(engagement_conversion.index.astype(str), engagement_conversion.values, marker='o', color=ACCENT_A,
        linewidth=2.2, markersize=6, markerfacecolor='white', markeredgewidth=1.5)
ax.fill_between(range(len(engagement_conversion)), engagement_conversion.values, color=ACCENT_A, alpha=0.12)
ax.set_title('Conversion Rate by Engagement Level')
ax.set_ylabel('Conversion Rate (%)')
plt.tight_layout()
plt.show()

In [ ]:
device_conversion = df.groupby('Device')['Converted'].mean().mul(100)

plt.figure(figsize=(7, 4.5))
ax = plt.gca()
ax.bar(device_conversion.index, device_conversion.values, color=CATEGORICAL[:len(device_conversion)],
       edgecolor='white', linewidth=0.6, width=0.55)
ax.set_title('Conversion Rate by Device')
ax.set_ylabel('Conversion Rate (%)')
add_bar_labels(ax, "{:.1f}%")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.boxplot(data=df, x='Converted', y='Ad_Spend', hue='Converted',
                  palette=GOOD_BAD[::-1], legend=False)
ax.set_title('Advertising Spend by Conversion Status')
ax.set_xlabel('Converted (0 = No, 1 = Yes)')
ax.set_xticklabels(['No', 'Yes'])
plt.tight_layout()
plt.show()

## 7. Statistical Analysis

In [12]:
converted_spend = df.loc[df['Converted']==1, 'Ad_Spend']
not_converted_spend = df.loc[df['Converted']==0, 'Ad_Spend']

t_stat, p_value = stats.ttest_ind(
    converted_spend,
    not_converted_spend,
    equal_var=False
)

print(f"T-statistic: {t_stat:.3f}")
print(f"P-value: {p_value:.5f}")
print("Statistically significant." if p_value < 0.05 else "Not statistically significant.")

T-statistic: -1.071
P-value: 0.28451
Not statistically significant.


## 8. KPI Summary

In [13]:
total_spend = df['Ad_Spend'].sum()
conversions = df['Converted'].sum()

pd.Series({
    'Leads': df['Lead_ID'].nunique(),
    'Conversions': conversions,
    'Conversion Rate (%)': df['Converted'].mean()*100,
    'Total Ad Spend': total_spend,
    'Cost per Conversion': total_spend/conversions if conversions else np.nan
}).round(2)

Leads                     6500.00
Conversions               1259.00
Conversion Rate (%)         19.37
Total Ad Spend         1302295.92
Cost per Conversion       1034.39
dtype: float64

## 9. Business Insights

- Compare channel conversion rates before reallocating marketing budget.
- Identify high-engagement segments with stronger conversion performance.
- Compare device behavior for campaign targeting.
- Track cost per conversion as a practical efficiency KPI.

## 10. Executive Dashboard

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
fig.suptitle('Marketing Campaign Analytics Dashboard', fontsize=18, fontweight='bold')

axes[0, 0].bar(channel_conversion.index, channel_conversion.values,
               color=SEQUENTIAL(np.linspace(0.25, 0.9, len(channel_conversion))), edgecolor='white', linewidth=0.6)
axes[0, 0].set_title('Conversion Rate by Channel')
axes[0, 0].tick_params(axis='x', rotation=20)

axes[0, 1].bar(channel_spend.index, channel_spend.values,
               color=CATEGORICAL[:len(channel_spend)], edgecolor='white', linewidth=0.6)
axes[0, 1].set_title('Ad Spend by Channel')
axes[0, 1].tick_params(axis='x', rotation=20)

axes[1, 0].plot(engagement_conversion.index.astype(str), engagement_conversion.values, marker='o',
                color=ACCENT_A, linewidth=2.2, markersize=6, markerfacecolor='white', markeredgewidth=1.5)
axes[1, 0].fill_between(range(len(engagement_conversion)), engagement_conversion.values, color=ACCENT_A, alpha=0.12)
axes[1, 0].set_title('Conversion by Engagement')

axes[1, 1].bar(device_conversion.index, device_conversion.values,
               color=CATEGORICAL[:len(device_conversion)], edgecolor='white', linewidth=0.6, width=0.55)
axes[1, 1].set_title('Conversion by Device')

for ax in axes.flat:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()